# Kaggle 通用训练模板（固定 revision）

设置 `DL_HELPER_GIT_REPO` 与 `DL_HELPER_GIT_REF`（40 位 commit SHA），然后运行 bootstrap。

In [ ]:
import os
os.environ['DL_HELPER_GIT_REPO'] = 'https://github.com/your-org/dl-helper.git'
os.environ['DL_HELPER_GIT_REF'] = '0000000000000000000000000000000000000000'  # 替换为固定 SHA
os.environ['DL_HELPER_MNIST_PATH'] = '/kaggle/input/<dataset>/mnist.npz'  # 替换为实际挂载路径

In [ ]:
import os
import subprocess
import sys

repo_url = os.environ['DL_HELPER_GIT_REPO']
revision = os.environ['DL_HELPER_GIT_REF']
repo_dir = '/kaggle/working/dl-helper'
if repo_url == 'https://github.com/your-org/dl-helper.git' or revision == '0' * 40:
    raise ValueError('请先填写真实 DL_HELPER_GIT_REPO 和 40 位 DL_HELPER_GIT_REF')
if os.path.exists(repo_dir):
    raise RuntimeError(f'拒绝复用未经校验的 checkout: {repo_dir}')

def run_checked(command, *, cwd=None):
    proc = subprocess.run(command, cwd=cwd, capture_output=True, text=True, encoding='utf-8')
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr, file=sys.stderr)
        raise SystemExit(proc.returncode)
    return proc

run_checked(['git', 'clone', repo_url, repo_dir])
run_checked(['git', 'checkout', revision], cwd=repo_dir)
head = run_checked(['git', 'rev-parse', 'HEAD'], cwd=repo_dir).stdout.strip()
if head.lower() != revision.lower():
    raise RuntimeError(f'checkout HEAD 不匹配: {head}')
os.environ['DL_HELPER_REPO_DIR'] = repo_dir
print(f'固定 revision 已检出: {head}')

In [ ]:
import subprocess, sys
proc = subprocess.run([sys.executable, '/kaggle/working/dl-helper/envs/kaggle_bootstrap.py'],
                      capture_output=True, text=True, encoding='utf-8')
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise SystemExit(proc.returncode)

In [ ]:
import subprocess, sys
proc = subprocess.run([sys.executable, '-m', 'dl_helper.training.cli', 'doctor',
                       '--config', '/kaggle/working/dl-helper-doctor.yaml',
                       '--experiment', 'experiments.mnist:build_experiment'],
                      cwd='/kaggle/working/dl-helper',
                      capture_output=True, text=True, encoding='utf-8')
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise SystemExit(proc.returncode)